# 02 — Preprocessing & Feature Engineering

**Goal:** Convert raw TSV files into tensors ready for the model.
Produces a cached bundle (`cache/preprocessed.pkl`) and a GloVe
embedding matrix (`cache/embedding_matrix.npy`) so `03_training.ipynb`
runs without redoing this work.

In [1]:
import os, sys, pickle, time
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

import numpy as np
import pandas as pd

from data_loader import (
    download_mind, download_glove,
    load_news, load_behaviors,
    NewsTokenizer, load_glove_matrix,
    encode_all_news, build_train_samples, build_eval_samples,
)

os.makedirs("../cache", exist_ok=True)

## 1. Ensure data is downloaded

In [2]:
download_mind("../data")
# NOTE: GloVe is ~820MB zipped. Uncomment to download automatically,
# or manually place glove.6B.300d.txt in data/glove/.
# glove_path = download_glove("../data")
glove_path = "../data/glove/glove.6B.300d.txt"
print(f"GloVe path: {glove_path}")
print(f"Exists? {os.path.exists(glove_path)}")

[skip] train already present at ../data\MINDsmall_train
[skip] dev already present at ../data\MINDsmall_dev
GloVe path: ../data/glove/glove.6B.300d.txt
Exists? True


## 2. Load TSVs

In [3]:
train_news = load_news("../data/MINDsmall_train/news.tsv")
dev_news   = load_news("../data/MINDsmall_dev/news.tsv")
train_beh  = load_behaviors("../data/MINDsmall_train/behaviors.tsv")
dev_beh    = load_behaviors("../data/MINDsmall_dev/behaviors.tsv")

# Union news so dev candidates are always encodable
all_news = (
    pd.concat([train_news, dev_news], ignore_index=True)
      .drop_duplicates("news_id")
      .reset_index(drop=True)
)
print(f"Train news : {len(train_news):,}")
print(f"Dev news   : {len(dev_news):,}")
print(f"Union      : {len(all_news):,}")

Train news : 51,282
Dev news   : 51,282
Union      : 51,282


## 3. Build vocabulary from TRAIN titles only

We build the vocabulary only from training titles to avoid leakage, but we will
*encode* dev news titles with the same vocabulary (OOV tokens become `<UNK>`).

In [4]:
MAX_TITLE_LEN = 30
MIN_WORD_FREQ = 2

tokenizer = NewsTokenizer(max_title_len=MAX_TITLE_LEN, min_word_freq=MIN_WORD_FREQ)
tokenizer.build_vocab(train_news["title"].tolist())

[vocab] size = 20,773 (from 37,535 unique tokens, min_freq=2)


## 4. Build GloVe embedding matrix

In [5]:
if os.path.exists(glove_path):
    embedding_matrix = load_glove_matrix(glove_path, tokenizer.word2idx, embed_dim=300)
    np.save("../cache/embedding_matrix.npy", embedding_matrix)
    print(f"Saved embedding matrix: {embedding_matrix.shape}")
else:
    print("WARNING: GloVe file not found. Run download_glove() above, or the "
          "model will fall back to random initialization in training.")

[glove] matched 19,068/20,773 words (91.8% coverage)
Saved embedding matrix: (20773, 300)


## 5. Encode every news title

In [6]:
t0 = time.time()
news_encoded = encode_all_news(all_news, tokenizer)
print(f"Encoded {len(news_encoded):,} articles in {time.time()-t0:.1f}s")
print("Example encoding for first article:")
first_id = all_news['news_id'].iloc[0]
print(f"  {first_id} -> {news_encoded[first_id][:10]}... (len={len(news_encoded[first_id])})")

Encoded 51,282 articles in 1.6s
Example encoding for first article:
  N55528 -> [2, 3, 4, 5, 6, 7, 8, 6, 9, 7]... (len=30)


## 6. Build TRAIN samples (negative sampling)

In [7]:
NEG_K = 4
MAX_HISTORY = 50

train_samples = build_train_samples(
    train_beh, news_encoded, neg_k=NEG_K, max_history=MAX_HISTORY, seed=42
)

# Quick sanity check
ex = train_samples[0]
print(f"Example train sample:")
print(f"  history len = {len(ex['history'])}")
print(f"  candidates  = {len(ex['candidates'])} (1 positive + {NEG_K} negatives)")

[train samples] 177,375 built, 0 impressions skipped
Example train sample:
  history len = 9
  candidates  = 5 (1 positive + 4 negatives)


## 7. Build EVAL samples (full candidate lists)

In [8]:
eval_samples = build_eval_samples(dev_beh, news_encoded, max_history=MAX_HISTORY)

# Distribution of candidates per impression in eval
cand_counts = [len(s["candidates"]) for s in eval_samples]
print(f"Eval impressions        : {len(eval_samples):,}")
print(f"Avg candidates / imp    : {np.mean(cand_counts):.1f}")
print(f"Median candidates / imp : {np.median(cand_counts):.0f}")

[eval samples] 38,369 built (dropped: 873 zero-history, 0 bad-labels)
Eval impressions        : 38,369
Avg candidates / imp    : 37.1
Median candidates / imp : 24


## 8. Cache the full preprocessed bundle

In [9]:
bundle = {
    "tokenizer": tokenizer,
    "news_encoded": news_encoded,
    "train_beh": train_beh,
    "dev_beh": dev_beh,
    "all_news": all_news,
}
with open("../cache/preprocessed.pkl", "wb") as f:
    pickle.dump(bundle, f)

# Save samples separately so you can rebuild training samples with a
# different seed or neg_k without re-tokenizing everything.
with open("../cache/train_samples.pkl", "wb") as f:
    pickle.dump(train_samples, f)
with open("../cache/eval_samples.pkl", "wb") as f:
    pickle.dump(eval_samples, f)

print("Preprocessing cache written to ../cache/")
for fn in os.listdir("../cache"):
    p = os.path.join("../cache", fn)
    print(f"  {fn:30s}  {os.path.getsize(p)/1e6:8.2f} MB")

Preprocessing cache written to ../cache/
  embedding_matrix.npy               24.93 MB
  eval_samples.pkl                   24.44 MB
  preprocessed.pkl                  145.18 MB
  train_samples.pkl                  35.08 MB
